# DLC Inference (O2)

Generate scripts, configs, and commands for running DLC inference on HMS O2.

**Local actions** (video discovery, YAML writing, script generation) execute
directly in this notebook. **SSH/rsync/sbatch commands** are printed for you
to paste into your Terminal or O2 SSH session — no SSH is executed from
within this notebook.

See `src/stroke_orofacial/dlc_inference_o2/README.md` for the full workflow
and troubleshooting guide.

In [ ]:
# ============================================================
# USER INPUTS — edit variables below, then run all cells.
# ============================================================

# Your HMS O2 username
USERNAME = "ps150"

# Animals and dates to process
ANIMALS = {
    "PS46": {"include": "all", "exclude": []},
    # "PS47": {"include": "all", "exclude": []},
    # "PS51": {"include": ["20250416", "20250417"], "exclude": ["20250426"]},
}

# Selection YAML filename (uploaded to O2)
SELECTION_YAML_NAME = "video_selection.yaml"

# Set True for a quick test on one video
TEST_MODE = False

In [ ]:
from pathlib import Path

from stroke_orofacial.config_loader import load_defaults
from stroke_orofacial.dlc_inference_o2.runner import generate_inference_script
from stroke_orofacial.dlc_inference_o2.batch import write_selection_yaml
from stroke_orofacial.dlc_inference_o2.o2_job import generate_sbatch_script, o2_paths
from stroke_orofacial.dlc_inference_o2.transfer import (
    build_stage_videos_command,
    build_copy_to_microscope_command,
    build_rename_command,
)

defaults = load_defaults()
config = defaults["dlc_inference_o2"]
paths = o2_paths(config, USERNAME)

print("O2 paths:")
for k, v in paths.items():
    print(f"  {k}: {v}")

## Step 1: Generate and save scripts locally

In [ ]:
# Write the inference runner script locally
local_scripts = Path("_dlc_inference_scripts")  # gitignored
local_scripts.mkdir(exist_ok=True)

# run_dlc_batch.py
runner_path = local_scripts / "run_dlc_batch.py"
runner_path.write_text(generate_inference_script())
print(f"Wrote: {runner_path}")

# SBATCH script
sbatch_path = local_scripts / "run_dlc.sbatch"
sbatch_path.write_text(generate_sbatch_script(config, USERNAME, SELECTION_YAML_NAME))
print(f"Wrote: {sbatch_path}")

# Video selection YAML
yaml_path = local_scripts / SELECTION_YAML_NAME
write_selection_yaml(yaml_path, ANIMALS)
print(f"Wrote: {yaml_path}")

## Step 2: Upload scripts to O2

**Paste these commands into your Terminal** (not in this notebook):

In [ ]:
print(f"""# Upload scripts and config to O2
ssh {USERNAME}@o2.hms.harvard.edu "mkdir -p {paths['dlc_code']} {paths['dlc_cfg']} {paths['log_dir']}"

scp {runner_path} {USERNAME}@o2.hms.harvard.edu:{paths['dlc_code']}/run_dlc_batch.py
scp {sbatch_path} {USERNAME}@o2.hms.harvard.edu:{paths['dlc_code']}/run_dlc.sbatch
scp {yaml_path} {USERNAME}@o2.hms.harvard.edu:{paths['dlc_cfg']}/{SELECTION_YAML_NAME}
""")

## Step 3: Stage videos to O2 scratch

**Paste into your O2 SSH session:**

In [ ]:
all_dates = sorted({d for spec in ANIMALS.values() for d in (spec.get("include", []) if spec.get("include") != "all" else [])})
if not all_dates:
    print("# NOTE: 'include: all' specified — list dates explicitly for staging,")
    print("# or rsync the entire camera tree (slow). Example:")
    all_dates = ["20250307"]  # placeholder

cmd = build_stage_videos_command(config, USERNAME, list(ANIMALS.keys()), all_dates)
print(cmd)

## Step 4: Submit the SBATCH job

**Paste into your O2 SSH session:**

In [ ]:
print(f"""# Submit DLC inference job
sbatch {paths['dlc_code']}/run_dlc.sbatch

# Monitor
squeue -u {USERNAME}
# tail -f {paths['log_dir']}/dlc_<JOBID>.out
""")

## Step 5: Post-processing (after job completes)

### 5a: Rename snapshots
**Paste into O2 SSH session:**

In [ ]:
print(build_rename_command(config, USERNAME))

### 5b: Copy outputs to MICROSCOPE
**Paste into Terminal (two-stage recommended):**

In [ ]:
print(build_copy_to_microscope_command(config, USERNAME, via_data1=True))